# Generated qLDLC decoding

Load one compact, documented generated-code fixture. Belief propagation uses an overcomplete graph while hard decisions use the verified square reduced basis.

In [ ]:
import Pkg
repo_root = isfile(joinpath(pwd(), "Project.toml")) ? pwd() : normpath(joinpath(pwd(), ".."))
Pkg.activate(repo_root)

using LinearAlgebra
using LatticeDecoder
using Random

samples_per_point = parse(Int, get(ENV, "LATTICEDECODER_EXAMPLE_SAMPLES", "20"))

using NPZ

In [ ]:
fixture_path = joinpath(repo_root, "examples", "fixtures", "qldlc_n13_3.npz")
data = npzread(fixture_path)
H = data["bp_H"]
decision_H = data["decision_H"]
G = data["G"]
logical_check = data["logical_check"]

problem = QuantumDecodingProblem(H, G, logical_check; decision_H)
sigma = 0.10
rng = MersenneTwister(50)
error_vector = sample_error(rng, sigma, size(H, 2))
received = copy(error_vector)
decoder = LDLCDecoder(
    initialize_tanner_graph(H);
    schedule=:serial,
    algorithm=:lsd,
    sigma,
    max_iterations=6,
)
soft_estimate = run_decoder!(decoder, received)
decision = hard_decision(soft_estimate, decision_H)
residual = error_vector - (received - G * decision)

(
    bp_graph_size=size(H),
    decision_matrix_size=size(decision_H),
    logical_error=is_logical_error(logical_check, residual),
)

In [ ]:
sigmas = [0.08, 0.10, 0.12]
estimates = [
    estimate_logical_error_rate!(
        MersenneTwister(500),
        LDLCDecoder(
            initialize_tanner_graph(H);
            schedule=:serial,
            algorithm=:lsd,
            sigma,
            max_iterations=6,
        ),
        problem;
        samples=samples_per_point,
    )
    for sigma in sigmas
]

[
    (
        sigma=sigma,
        failures=result.events,
        samples=result.samples,
        rate=result.rate,
        interval=(result.lower, result.upper),
    )
    for (sigma, result) in zip(sigmas, estimates)
]

The original generated-code collection is not part of the public repository. The retained fixture is sufficient to reproduce this API workflow and is described in `examples/fixtures/README.md`.